In [1]:
import subprocess, textwrap, json

filename = "0_floating_arch.js"

js_wrapper = textwrap.dedent("""
const fs = require('fs');
const code = fs.readFileSync('""" + filename + """','utf8');

// --- expose mocks on GLOBAL scope so eval'ed function always sees them ---
global.bot = {
  interrupt_code: false,
  // arch.js가 필요로 하는 필드
  entity: { position: { x: 0, y: 0, z: 0 } }
};

global.world = {
  // arch_differentiable.js가 필요로 하는 API
  getPosition: (_bot) => ({ x: 0, y: 0, z: 0 }),
  // 청소 단계에서 무엇을 반환하든 placeBlock 로깅에는 직접 영향 없음
  // (다만 분기 로직을 타게 하려면 air가 아닌 것도 테스트해볼 수 있음)
  getBlockAtPosition: (_bot, _x, _y, _z) => ({ name: 'air' })
};

global.log = function () {}; // 더미

// placeBlock 후킹을 전역으로
global.skills = {
  breakBlockAt: async () => {},
  placeBlock: async (_bot, block, x, y, z, ...rest) => {
    console.log(JSON.stringify({ x, y, z, material: block }));
  }
};

(async () => {
  try {
    const evaluated = eval(code);             // (async (bot)=>{...}) 형태 기대
    const fn = (typeof evaluated === 'function')
      ? evaluated
      : (evaluated && typeof evaluated.default === 'function' ? evaluated.default : null);

    if (fn) {
      await fn(global.bot);                   // 실제 실행
    } else {
      // 혹시 함수가 아니라 즉시실행(IIFE) 같은 형태면 여기서 아무 것도 안 할 수 있음
      // 필요하면 추가 처리
    }
  } catch (e) {
    // 에러를 stderr로 넘겨서 파이썬에서 확인 가능
    console.error("WRAPPER_ERROR:", e && e.stack || e);
  }
})();
""")

res = subprocess.run(["node", "-e", js_wrapper], capture_output=True, text=True)

coords = []
for line in res.stdout.splitlines():
    line = line.strip()
    if not line: 
        continue
    try:
        coords.append(json.loads(line))
    except json.JSONDecodeError:
        pass

print("logged:", len(coords))
print(coords[:5])


logged: 251
[{'x': -6, 'y': 0, 'z': -1, 'material': 'oak_planks'}, {'x': -6, 'y': 0, 'z': 0, 'material': 'oak_planks'}, {'x': -6, 'y': 0, 'z': 1, 'material': 'oak_planks'}, {'x': -5, 'y': 0, 'z': -1, 'material': 'oak_planks'}, {'x': -5, 'y': 0, 'z': 0, 'material': 'oak_planks'}]


In [2]:
import plotly.graph_objects as go

# ── 재료별 색상 (원하는 대로 추가/수정 가능) ──
MAT_COLOR = {
    "stone": "#888888",
    "stone_bricks": "#9aa0a6",
    "dirt": "#8B4513",
    "oak_planks": "#b8860b",
    "oak_fence": "#a0522d",
    "stone_slab": "#b0b0b0",
    "torch": "#f1c40f",
}

def cube_vertices(x, y, z, size=1):
    return [
        (x,     y,     z), (x+size, y,     z), (x+size, y+size, z), (x,     y+size, z),
        (x,     y, z+size), (x+size, y, z+size), (x+size, y+size, z+size), (x,     y+size, z+size)
    ]

def plot_blocks_cubes(coords, title="Blocks (Plotly Mesh3d)", alpha=0.5):
    X, Y, Z = [], [], []
    I, J, K = [], [], []
    facecols = []

    faces = [
        (0,1,2,3), (4,5,6,7), (0,1,5,4),
        (2,3,7,6), (1,2,6,5), (0,3,7,4)
    ]

    for c in coords:
        x, y, z = int(c["x"]), int(c["y"]), int(c["z"])
        mat = c.get("material", "stone")
        color = MAT_COLOR.get(mat, "#000000")  # ← 없는 블록은 검정색으로

        verts = cube_vertices(x, y, z, size=1)
        base = len(X)

        for vx, vy, vz in verts:
            X.append(vx)     # 가로(X)
            Y.append(vz)     # 깊이(MC z → Plotly Y축)
            Z.append(vy)     # 높이(MC y → Plotly Z축)

        for f in faces:
            i0,i1,i2,i3 = [base+fi for fi in f]
            I += [i0, i0]
            J += [i1, i2]
            K += [i2, i3]
            facecols += [color, color]

    fig = go.Figure(data=[go.Mesh3d(
        x=X, y=Y, z=Z, i=I, j=J, k=K,
        facecolor=facecols, opacity=alpha, flatshading=True
    )])

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="X",
            yaxis_title="Z",
            zaxis_title="Y (Height)",
            aspectmode="data"
        )
    )
    fig.show()

# 사용 예:
plot_blocks_cubes(coords)

In [3]:
from collections import deque

def _toi(v): return int(str(v).strip())

def has_no_floating_blocks(coords, ground_y=0):
    """
    모든 블록이 y==ground_y 인 '지면'과 연결돼 있으면 True,
    하나라도 연결되지 못하면 False를 반환.
    각 coord(dict)에 visit 플래그를 추가/갱신한다.
    coords: [{"x":..,"y":..,"z":..}, ...]
    """
    if not coords:
        return True  # 비어있으면 떠 있는 것도 없음

    # 정규화 + 위치 인덱스 맵
    pos_to_idx = {}
    for i, c in enumerate(coords):
        c["x"], c["y"], c["z"] = _toi(c["x"]), _toi(c["y"]), _toi(c["z"])
        c["visit"] = False
        pos_to_idx[(c["x"], c["y"], c["z"])] = i

    # 지면(y==ground_y)과 맞닿은 블록들로부터 BFS 시작
    q = deque()
    for i, c in enumerate(coords):
        if c["y"] == ground_y:
            c["visit"] = True
            q.append((c["x"], c["y"], c["z"]))

    # 지면 블록이 하나도 없으면 모두 붕 떠 있음으로 간주
    if not q:
        return False

    # 6-이웃(면 공유) 탐색
    NEI = [(1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)]
    while q:
        x, y, z = q.popleft()
        for dx, dy, dz in NEI:
            npos = (x+dx, y+dy, z+dz)
            j = pos_to_idx.get(npos)
            if j is not None and not coords[j]["visit"]:
                coords[j]["visit"] = True
                q.append(npos)

    # 방문 못 한 블록(=떠 있는 블록)이 있으면 False
    for c in coords:
        if not c["visit"]:
            return False
    return True


In [4]:
def _to_int(v): return int(str(v).strip())

def _xyz_list(coords):
    xs, ys, zs = [], [], []
    for c in coords:
        x, y, z = _to_int(c["x"]), _to_int(c["y"]), _to_int(c["z"])
        xs.append(x); ys.append(y); zs.append(z)
    return xs, ys, zs

def eval(coords):
    xs, ys, zs = _xyz_list(coords)
    if not coords:
        return False, "Count mismatch: 0 (expected >0)"

    mean_y = sum(ys) / len(ys)
    minx, maxx = min(xs), max(xs)
    minz, maxz = min(zs), max(zs)
    cx = round((minx + maxx) / 2)
    cz = round((minz + maxz) / 2)

    # 1) 중심선(x=cx, z=cz)에서의 최대 높이가 전체 평균 이상인가?
    max_y_on_center_x = max(y for x,y in zip(xs,ys) if x == cx)
    max_y_on_center_z = max(y for z,y in zip(zs,ys) if z == cz)
    cond1 = (max_y_on_center_x >= mean_y) and (max_y_on_center_z >= mean_y)

    # 2) 정확한 중심 기둥(x=cx AND z=cz)의 최대 y가 평균 이상인가?
    center = tuple((sum(c[k] for c in coords)/len(coords)) for k in ("x","y","z"))    
    max_y = max((c["y"] for c in coords if c["x"]==round(center[0]) and c["z"]==round(center[2])), default=0)
    cond2 = max_y > center[1]

    # 3) 가장자리(x=min/max, z=min/max)의 y가 평균 미만인가?
    edge_y = [y for x,y in zip(xs,ys) if x in (minx,maxx)]
    edge_y += [y for z,y in zip(zs,ys) if z in (minz,maxz)]
    cond3 = max_y > (sum(edge_y)/len(edge_y))

    # 4) 둥둥 떠있는 블록 체크
    cond4 = has_no_floating_blocks(coords)

    ok = cond1 and cond2 and cond3 and cond4
    return ok, f"arch={ok} | {cond1} {cond2} {cond3} {cond4}"


# 사용 예:
ok, msg = eval(coords)
print(ok, msg)


False arch=False | True True True False
